In [2]:
# Cell 1 - import wing-x for Light Jets
import pandas as pd
import numpy as np

# 1. SETUP & LOAD
METRO_PATH = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro.parquet"
print("🚀 Loading and processing Light Jet market audit...")
df = pd.read_parquet(METRO_PATH)

# 2. STANDARDIZE & FILTER
df['FlightDate_ET'] = pd.to_datetime(df['FlightDate_ET'])
df = df.rename(columns={'FromCluster': 'FromMetro', 'ToCluster': 'ToMetro'})

# Define the specific LJ models requested
lj_models = ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra']

# --- LJ SPECIFIC FILTER ---
mask = (
    (df['aircraft_segment'] == 'Light Jet') & 
    (df['aircraft_model'].isin(lj_models)) & 
    (df['FromMetro'] != 'OTHER_METRO') & 
    (df['ToMetro'] != 'OTHER_METRO') & 
    (df['FromMetro'] != df['ToMetro']) & # Kills Intra-Metro ghosts
    (df['Hours'] >= 1.5) # Adjusted threshold for Light Jets (e.g., 1.5 hrs)
)

# Create lj_lh as a clean, independent copy
lj_lh = df[mask].copy()

# Define corridor only for legitimate inter-metro pairs
lj_lh['corridor'] = lj_lh['FromMetro'] + " ➔ " + lj_lh['ToMetro']

# 3. WUP PRESENCE DEFINITION
wup_mask = ((lj_lh['Operator'].str.contains('Wheels Up', case=False, na=False)) | 
            (lj_lh['Operator'] == 'Mountain Aviation'))
lj_lh['is_wup'] = wup_mask

# 4. BI-DIRECTIONAL AGGREGATION
def make_pair_key(row):
    return " <-> ".join(sorted([row['FromMetro'], row['ToMetro']]))

lj_lh['pair_key'] = lj_lh.apply(make_pair_key, axis=1)

# Aggregation logic for audit
pair_analysis = lj_lh.groupby('pair_key').agg(
    total_pair_missions=('pair_key', 'size'),
    total_wup_missions=('is_wup', 'sum'),
    avg_hours_combined=('Hours', 'mean')
).reset_index()

def get_splits(group):
    counts = group['corridor'].value_counts()
    hrs = group.groupby('corridor')['Hours'].mean()
    wup = group.groupby('corridor')['is_wup'].sum()
    total = group.groupby('corridor').size()
    return pd.Series({
        'mission_split': " | ".join(counts.astype(str)),
        'hours_split': " | ".join([f"{h:.2f}" for h in hrs]),
        'share_split': " | ".join([f"{(w/t)*100:.1f}%" for w, t in zip(wup, total)])
    })

# 5. VALIDATION & OUTPUT
splits = lj_lh.groupby('pair_key').apply(get_splits).reset_index()
pair_analysis = pair_analysis.merge(splits, on='pair_key').sort_values('total_pair_missions', ascending=False)

# Validation print
intra_metro_count = lj_lh[lj_lh['FromMetro'] == lj_lh['ToMetro']].shape[0]
print(f"✅ LJ Market Audit Complete.")
print(f"1. Total Inter-Metro Missions: {len(lj_lh):,}")
print(f"2. Unique Directional Corridors: {lj_lh['corridor'].nunique()}")
print(f"3. Intra-Metro (Same City) Missions: {intra_metro_count}")
print(f"  Date range: {df['FlightDate_'].min().date()} to {df['FlightDate'].max().date()}")

display(pair_analysis.head(10))

🚀 Loading and processing Light Jet market audit...
✅ LJ Market Audit Complete.
1. Total Inter-Metro Missions: 102,664
2. Unique Directional Corridors: 430
3. Intra-Metro (Same City) Missions: 0


/var/tmp/ipykernel_60430/2363278605.py:63: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  splits = lj_lh.groupby('pair_key').apply(get_splits).reset_index()


KeyError: 'FlightDate'

In [15]:
# -----------------------------------------------------------------------------------------
# CELL 2: DIRECTIONAL SEASONAL DISCOVERY — LIGHT JETS
# -----------------------------------------------------------------------------------------
from sklearn.cluster import KMeans

# 1. Build monthly corridor fingerprint
seasonal_counts = (lj_lh.groupby(["corridor", "month"]).size().unstack(fill_value=0))
seasonal_counts = seasonal_counts.reindex(columns=range(1, 13), fill_value=0)

# 2. Filter for statistical signal (Lowered to 20 for LJ)
MIN_MISSIONS = 20
annual_missions = seasonal_counts.sum(axis=1)
relevant_corridors = annual_missions > MIN_MISSIONS
seasonal_counts_relevant = seasonal_counts.loc[relevant_corridors].copy()

# 3. Normalize to monthly density (isolates the "shape")
monthly_density = seasonal_counts_relevant.div(seasonal_counts_relevant.sum(axis=1), axis=0)

# 4. Create seasonality index (1.00x = Average Month)
seasonality_index = monthly_density / (1 / 12)

# 5. K-Means clustering (Locked to 4 clusters per SMID logic)
km = KMeans(n_clusters=4, random_state=42, n_init=50)
cluster_ids = km.fit_predict(monthly_density)

# 6. Build audit/output dataframe
X_discovery = monthly_density.copy()
X_discovery["cluster_id"] = cluster_ids
X_discovery["annual_missions"] = annual_missions.loc[relevant_corridors] / 2

for m in range(1, 13):
    X_discovery[f"seasonality_index_m{m}"] = seasonality_index[m]

# 7. Display Heatmap 1: Monthly Density (%)
cluster_centers_density = (X_discovery.groupby("cluster_id")[list(range(1, 13))].mean().reindex(columns=range(1, 13)))
print(f"✅ Processed {len(X_discovery)} LJ corridors into 4 seasonal demand regimes.")
print("\n📈 MONTHLY DENSITY BY CLUSTER:")
display(cluster_centers_density.style.background_gradient(axis=1, cmap="YlOrRd").format("{:.1%}"))

# 8. Display Heatmap 2: Seasonality Index (x)
seasonality_cols = [f"seasonality_index_m{m}" for m in range(1, 13)]
cluster_centers_index = (X_discovery.groupby("cluster_id")[seasonality_cols].mean())
cluster_centers_index.columns = range(1, 13)
print("\n📈 SEASONALITY INDEX BY CLUSTER:")
display(cluster_centers_index.style.background_gradient(axis=1, cmap="YlOrRd").format("{:.2f}x"))

✅ Processed 360 LJ corridors into 4 seasonal demand regimes.

📈 MONTHLY DENSITY BY CLUSTER:


month,1,2,3,4,5,6,7,8,9,10,11,12
cluster_id,,,,,,,,,,,,
0,11.0%,10.4%,10.8%,10.2%,9.1%,5.2%,4.9%,4.3%,5.4%,7.7%,10.3%,10.6%
1,5.5%,5.2%,5.0%,5.8%,8.9%,10.2%,16.3%,11.4%,11.9%,8.2%,6.1%,5.6%
2,4.3%,4.7%,5.7%,7.5%,8.9%,11.2%,7.9%,10.6%,11.8%,12.5%,9.6%,5.3%
3,7.2%,7.4%,8.1%,9.2%,9.3%,7.9%,7.6%,8.2%,8.3%,10.1%,8.4%,8.5%



📈 SEASONALITY INDEX BY CLUSTER:


,1,2,3,4,5,6,7,8,9,10,11,12
cluster_id,,,,,,,,,,,,
0,1.32x,1.25x,1.29x,1.22x,1.09x,0.63x,0.59x,0.52x,0.65x,0.92x,1.24x,1.28x
1,0.66x,0.62x,0.59x,0.69x,1.07x,1.23x,1.95x,1.37x,1.42x,0.99x,0.73x,0.67x
2,0.52x,0.56x,0.68x,0.90x,1.07x,1.34x,0.95x,1.27x,1.42x,1.51x,1.15x,0.64x
3,0.86x,0.89x,0.97x,1.10x,1.11x,0.95x,0.91x,0.98x,1.00x,1.21x,1.01x,1.02x


In [16]:
# -----------------------------------------------------------------------------------------
# CELL 3.5: LABELED SEASONALITY AUDIT (SYNCED)
# -----------------------------------------------------------------------------------------

# 1. Re-define labels to ensure parity with Cell 4 Master Export
final_labels = {
    0: "winter-spring-peak",   
    1: "summer-peak",        
    2: "core-utilization", 
    3: "multi-peak-directional"
}

# 2. Calculate centers from X_discovery (Normalized Density)
labeled_centers = X_discovery.groupby("cluster_id")[list(range(1, 13))].mean()

# 3. Apply labels to the index for the Heatmap display
labeled_centers.index = [f"{i}: {final_labels.get(i, 'Unknown')}" for i in labeled_centers.index]

print("🎯 FINAL LJ ARCHETYPE HEATMAP (Monthly Density %):")
display(
    labeled_centers.style.background_gradient(axis=1, cmap="YlOrRd")
    .format("{:.1%}")
)

# 4. Toggle Logic Validation (May vs June)
# Index 5 = May, Index 6 = June
summer_jump = labeled_centers[6] - labeled_centers[5]
top_toggle = summer_jump.idxmax()

print(f"\n💡 The strongest June 'Toggle' candidate is: {top_toggle}")
print(f"📈 Momentum: +{summer_jump.max():.1%} absolute density increase MoM.")

🎯 FINAL LJ ARCHETYPE HEATMAP (Monthly Density %):


month,1,2,3,4,5,6,7,8,9,10,11,12
0: winter-spring-peak,11.0%,10.4%,10.8%,10.2%,9.1%,5.2%,4.9%,4.3%,5.4%,7.7%,10.3%,10.6%
1: summer-peak,5.5%,5.2%,5.0%,5.8%,8.9%,10.2%,16.3%,11.4%,11.9%,8.2%,6.1%,5.6%
2: core-utilization,4.3%,4.7%,5.7%,7.5%,8.9%,11.2%,7.9%,10.6%,11.8%,12.5%,9.6%,5.3%
3: multi-peak-directional,7.2%,7.4%,8.1%,9.2%,9.3%,7.9%,7.6%,8.2%,8.3%,10.1%,8.4%,8.5%



💡 The strongest June 'Toggle' candidate is: 2: core-utilization
📈 Momentum: +2.3% absolute density increase MoM.


In [17]:
# -----------------------------------------------------------------------------------------
# FINAL VALIDATION: LJ UNIVERSE & VOLUME DISTRIBUTION
# -----------------------------------------------------------------------------------------

# 1. Sync labels specifically for LJ Archetypes (Verified in Cell 3.5)
final_labels_lj = {
    0: "winter-spring-peak",   
    1: "summer-peak",        
    2: "core-utilization", 
    3: "multi-peak-directional"
}

# 2. Metadata & Volume Calculations
total_corridors = len(X_discovery)
total_flights = X_discovery['annual_missions'].sum()

# Grouping by mapped labels for volume analysis
stats = X_discovery.groupby('cluster_id').agg({'annual_missions': ['count', 'sum']})
stats.index = stats.index.map(final_labels_lj)
stats.columns = ['corridor_count', 'flight_volume']
stats['volume_pct'] = (stats['flight_volume'] / total_flights) * 100

# 3. Intra-Metro Check
intra_metro_issues = [c for c in X_discovery.index if c.split(' ➔ ')[0] == c.split(' ➔ ')[1]]

# 4. Validation Output
print("📊 --- LIGHT JET CLUSTER UNIVERSE SUMMARY ---")
print(f"📌 Filtering Criteria Applied:")
print(f"   - Segment      : Light Jet (Select Models)")
print(f"   - Min Leg Time : >= 1.5 Hours")
print(f"   - Relevance    : > {MIN_MISSIONS} missions total (2-yr)")
print(f"   - Logic        : Inter-Metro Only (No Same-City)")

print(f"\n✅ Total Inter-Metro Corridors : {total_corridors}")
print(f"✅ Total Annual Flight Volume  : {total_flights:,.0f} (Annualized /2)")

print("\n📍 LJ ARCHETYPE BREAKDOWN (Volume & Count):")
print(f"{'Archetype':<25} | {'Corridors':<10} | {'Flights':<12} | {'Volume %'}")
print("-" * 65)
for label, row in stats.sort_values('flight_volume', ascending=False).iterrows():
    print(f"{label:<25} | {int(row['corridor_count']):<10} | {int(row['flight_volume']):<12,} | {row['volume_pct']:.1f}%")

print("\n🛡️ --- SAFETY CHECKS ---")
if not intra_metro_issues:
    print(f"✅ Final Validation Passed: 0 intra-metro corridors found.")
else:
    print(f"❌ Final Validation Failed: Found {len(intra_metro_issues)} intra-metro records!")

📊 --- LIGHT JET CLUSTER UNIVERSE SUMMARY ---
📌 Filtering Criteria Applied:
   - Segment      : Light Jet (Select Models)
   - Min Leg Time : >= 1.5 Hours
   - Relevance    : > 20 missions total (2-yr)
   - Logic        : Inter-Metro Only (No Same-City)

✅ Total Inter-Metro Corridors : 360
✅ Total Annual Flight Volume  : 51,049 (Annualized /2)

📍 LJ ARCHETYPE BREAKDOWN (Volume & Count):
Archetype                 | Corridors  | Flights      | Volume %
-----------------------------------------------------------------
multi-peak-directional    | 147        | 22,416       | 43.9%
winter-spring-peak        | 88         | 17,122       | 33.5%
core-utilization          | 76         | 6,399        | 12.5%
summer-peak               | 49         | 5,111        | 10.0%

🛡️ --- SAFETY CHECKS ---
✅ Final Validation Passed: 0 intra-metro corridors found.
